# Ачча! — озвучка фраз (Indic Parler-TTS)

Блокнот озвучивает фразы приложения «Ачча!» открытой моделью **Indic Parler-TTS** (AI4Bharat, лицензия Apache 2.0) и сохраняет mp3 в ваш Google Drive.

**Перед запуском:** меню **Среда выполнения → Сменить среду выполнения → Аппаратный ускоритель: T4 GPU → Сохранить**.

Дальше запускайте ячейки по порядку (кнопка ▶ слева от ячейки). Шаг 5 — пробное прослушивание: если голос не понравился, поменяйте имя голоса на шаге 2 и запустите шаги 2 и 5 ещё раз. Шаг 6 озвучивает всё (10–20 минут).

## 1. Установка (2–3 минуты)

В конце сеанс перезапустится сам и появится сообщение о сбое — это нормально. Дальше начинайте с шага 2, шаг 1 повторять не нужно.

In [ ]:
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile
# parler-tts ставит старую версию protobuf, из-за неё не загружается transformers — возвращаем новую
!pip -q install -U "protobuf>=5.28"
!apt -qq install -y ffmpeg > /dev/null
print('Готово. Сеанс сейчас перезапустится сам — это нормально. После перезапуска начинайте с шага 2.')
import os; os.kill(os.getpid(), 9)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Настройки

Для английского создатели модели рекомендуют голоса **Thoma** (мужской) и **Mary** (женский); полный список из 21 голоса — на странице модели huggingface.co/ai4bharat/indic-parler-tts.
- `LOCAL_VOICE` — голос для реплик индийцев (индийский английский)
- `TOURIST_VOICE` — голос для реплик туриста
- `ONLY_LOCAL = True` — озвучить только реплики индийцев (туристу останется голос телефона)
- `FIX` — исправления произношения: слева как написано во фразе, справа как модели прочитать. Если слово звучит неправильно, добавьте его сюда
- `ONLY_IDS` — переозвучить только эти фразы, например `['air-03']`. Пустой список `[]` — озвучить всё


In [1]:
PHRASES_URL = 'https://achha-india.github.io/app/phrases.json'

LOCAL_VOICE = 'Thoma'
TOURIST_VOICE = 'Mary'
ONLY_LOCAL = False

LOCAL_STYLE = "{name} speaks in a clear Indian English accent at a moderate pace, with a friendly tone. The recording is very clear and close, with no background noise."
TOURIST_STYLE = "{name} speaks slowly and clearly with a neutral accent, in a calm tone. The recording is very clear and close, with no background noise."

FIX = {
    'e-visa': 'e-veeza',
}
ONLY_IDS = ['air-03']

OUT_DIR = '/content/drive/MyDrive/Achha/audio'
print('Настройки сохранены')

Настройки сохранены


## 3. Загрузка модели (3–5 минут)

Если появится ошибка про доступ (401 / gated), откройте страницу модели huggingface.co/ai4bharat/indic-parler-tts, войдите в бесплатный аккаунт Hugging Face, нажмите «Agree and access» и добавьте токен в Colab: значок ключа слева → `HF_TOKEN`.

In [2]:
import torch
from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get('HF_TOKEN'))
    print('Вход в Hugging Face выполнен')
except Exception as e:
    print('Не удалось прочитать секрет HF_TOKEN:', e)
    print('Проверьте: имя секрета ровно HF_TOKEN и включён «Доступ из блокнотов»')
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device, '(если cpu — включите T4 GPU, см. начало)')
MODEL_ID = 'ai4bharat/indic-parler-tts'
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SR = model.config.sampling_rate
print('Модель загружена')

Вход в Hugging Face выполнен


Устройство: cuda (если cpu — включите T4 GPU, см. начало)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Модель загружена


## 4. Фразы с сайта

In [3]:
import json, urllib.request, torch
if 'model' not in globals() or 'PHRASES_URL' not in globals():
    raise RuntimeError('Сначала запустите шаг 2 и шаг 3 (после перезапуска сеанса Colab всё забывает)')
data = json.load(urllib.request.urlopen(PHRASES_URL))
lines = [l for st in data['stages'] for s in st['situations'] for l in s['lines']]
todo = [l for l in lines if l['role'] == 'local' or not ONLY_LOCAL]
if ONLY_IDS: todo = [l for l in lines if l['id'] in ONLY_IDS]
print('Всего реплик:', len(lines), '| к озвучке:', len(todo))

import soundfile as sf, numpy as np, subprocess, os, re

def synth(text, role, seed=7):
    name = LOCAL_VOICE if role == 'local' else TOURIST_VOICE
    style = (LOCAL_STYLE if role == 'local' else TOURIST_STYLE).format(name=name)
    for k, v in FIX.items():
        text = re.sub(re.escape(k), v, text, flags=re.I)
    torch.manual_seed(seed)
    d = description_tokenizer(style, return_tensors='pt').to(device)
    p = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                           prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    a = g.cpu().numpy().squeeze().astype(np.float32)
    # обрезаем тишину по краям
    nz = np.where(np.abs(a) > 0.01)[0]
    if len(nz): a = a[max(0, nz[0] - int(0.05 * SR)): nz[-1] + int(0.15 * SR)]
    return a

def to_mp3(audio, path):
    wav = path.replace('.mp3', '.wav')
    sf.write(wav, audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-ac', '1', '-b:a', '64k', path], check=True)
    os.remove(wav)
print('Готово')

Всего реплик: 57 | к озвучке: 1
Готово


## 5. Пробное прослушивание

In [4]:
from IPython.display import Audio, display
samples = todo if ONLY_IDS else [l for l in todo if l['id'] in ('market-02', 'road-13', 'market-01')] or todo[:3]
for l in samples:
    print(l['id'], '·', 'индиец' if l['role'] == 'local' else 'турист', '·', l['en'])
    display(Audio(synth(l['en'], l['role']), rate=SR))

air-03 · индиец · Show your e-visa.


## 6. Озвучить всё и сохранить в Google Drive

Colab попросит доступ к Google Drive — разрешите. Файлы появятся в папке **Achha/audio**, а рядом архив **achha-audio.zip**. Этот архив пришлите в чат.

In [5]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
done, failed = 0, []
for i, l in enumerate(todo, 1):
    path = f"{OUT_DIR}/{l['id']}.mp3"
    if os.path.exists(path) and not ONLY_IDS:
        done += 1; continue
    try:
        to_mp3(synth(l['en'], l['role']), path)
        done += 1
        print(f"[{i}/{len(todo)}] {l['id']} ✓")
    except Exception as e:
        failed.append(l['id']); print(f"[{i}/{len(todo)}] {l['id']} ошибка: {e}")
import shutil
zip_path = shutil.make_archive('/content/drive/MyDrive/Achha/achha-audio', 'zip', OUT_DIR)
print('\nГотово:', done, '| ошибки:', failed or 'нет')
print('Архив:', zip_path)
from google.colab import files
files.download(zip_path)

Mounted at /content/drive
[1/1] air-03 ✓

Готово: 1 | ошибки: нет
Архив: /content/drive/MyDrive/Achha/achha-audio.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Переозвучить отдельные фразы:** удалите их mp3 из папки Achha/audio на Drive и запустите шаг 6 ещё раз — остальные файлы он пропустит. Можно также поменять `seed` в функции `synth` — интонация станет немного другой.

**Неправильно звучит слово:** добавьте его в `FIX` на шаге 2, впишите номер фразы в `ONLY_IDS`, запустите шаги 2, 4, 5 (проверить) и 6. Фраза перезапишется, остальные не тронутся.